## Mysql installation

In [36]:
%%dockerexec hadoop

# sudo apt update
# sudo apt install -qq -y mysql-server unzip >> /tmp/install.log 2>&1

# Enable external access (from worker nodes)
sudo sed -i "s/^bind-address/#bind-address/g" /etc/mysql/mysql.conf.d/mysqld.cnf 

sudo service mysql restart
sudo service mysql status

 * Stopping MySQL database server mysqld
   ...done.
 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.
 * /usr/bin/mysqladmin  Ver 8.0.42-0ubuntu0.20.04.1 for Linux on x86_64 ((Ubuntu))
Copyright (c) 2000, 2025, Oracle and/or its affiliates.

Oracle is a registered trademark of Oracle Corporation and/or its
affiliates. Other names may be trademarks of their respective
owners.

Server version		8.0.42-0ubuntu0.20.04.1
Protocol version	10
Connection		Localhost via UNIX socket
UNIX socket		/var/run/mysqld/mysqld.sock
Uptime:			1 sec

Threads: 2  Questions: 8  Slow queries: 0  Opens: 119  Flush tables: 3  Open tables: 38  Queries per second avg: 8.000


In [37]:
%%dockerexec hadoop

# create hadoop user
sudo mysql -e "create user 'hadoop'@'%' IDENTIFIED BY ''"
sudo mysql -e "grant all privileges on *.* to 'hadoop'@'%' WITH GRANT OPTION"
sudo mysql -e "flush privileges"


ERROR 1396 (HY000) at line 1: Operation CREATE USER failed for 'hadoop'@'%'


### Airlines database setup

In [34]:
%%dockerexec hadoop

mysql -u hadoop -h hadoop < /opt/datasets/airlines.sql

Could not open connection to the HS2 server. Please check the server URI and if the URI is correct, then ask the administrator to check the server status.
Error: Could not open client transport with JDBC Uri: jdbc:hive2://localhost:10000: java.net.ConnectException: Connection refused (Connection refused) (state=08S01,code=0)


No current connection


In [38]:
%%dockerexec hadoop

mysql -u hadoop -h hadoop -e 'show databases'

printf "\n%40s\n\n" | tr ' ' '='

mysql -u hadoop -h hadoop -D airlines -e 'show tables'

Database
airlines
information_schema
mysql
performance_schema
sys


Tables_in_airlines
air_airlines
air_airplane_types
air_airplanes
air_airports
air_airports_geo
air_bookings
air_flights
air_flights_schedules
air_passengers
air_passengers_details


## Spark Setup

- version 3.5.0 (Pre-built for Apache Hadoop 3.3 and later)

In [15]:
%%dockerexec hadoop

# Download package
mkdir -p /opt/pkgs
cd /opt/pkgs
wget -q -c https://dlcdn.apache.org/spark/spark-3.5.6/spark-3.5.6-bin-hadoop3.tgz

# unpack file and create link
tar -zxf spark-3.5.6-bin-hadoop3.tgz -C /opt
ln -s /opt/spark-3.5.6-bin-hadoop3 /opt/spark

# update envvars.sh
cat >> /opt/envvars.sh << EOF
# Spark
export SPARK_HOME=/opt/spark
export PYSPARK_PYTHON=python3
export PYSPARK_DRIVER_PYTHON=python3
export PYTHONIOENCODING=utf8
export PATH=\${PATH}:\${SPARK_HOME}/bin

EOF

cat /opt/envvars.sh

ln: failed to create symbolic link '/opt/spark': File exists
export JAVA_HOME=/usr/lib/jvm/java-1.8.0-openjdk-amd64
export PDSH_RCMD_TYPE=ssh
export HADOOP_HOME=/opt/hadoop
export HADOOP_VERSION=3.3.6
export HADOOP_COMMON_HOME=${HADOOP_HOME}
export HADOOP_CONF_DIR=${HADOOP_HOME}/etc/hadoop
export HADOOP_HDFS_HOME=${HADOOP_HOME}
export HADOOP_MAPRED_HOME=${HADOOP_HOME}
export HADOOP_YARN_HOME=${HADOOP_HOME}
export PATH=${PATH}:${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin
# Spark
export SPARK_HOME=/opt/spark
export PYSPARK_PYTHON=python3
export PYSPARK_DRIVER_PYTHON=python3
export PYTHONIOENCODING=utf8
export PATH=${PATH}:${SPARK_HOME}/bin



In [18]:
%%dockerexec hadoop

source /opt/envvars.sh

mkdir -p /opt/src/spark

### Setup Sqoop

- download from https://archive.apache.org/dist/sqoop/1.4.7/
- version 1.4.7

In [50]:
%%dockerexec hadoop

source /opt/envvars.sh

# Download package
mkdir /opt/pkgs
cd /opt/pkgs
# wget -q -c https://downloads.apache.org/sqoop/1.4.7/sqoop-1.4.7.bin__hadoop-2.6.0.tar.gz
wget -q -c http://archive.apache.org/dist/sqoop/1.4.7/sqoop-1.4.7.bin__hadoop-2.6.0.tar.gz
    
# unpack file and create link
tar -zxf sqoop-1.4.7.bin__hadoop-2.6.0.tar.gz -C /opt
ln -s /opt/sqoop-1.4.7.bin__hadoop-2.6.0 /opt/sqoop

# update commons-lang
rm /opt/sqoop/lib/commons-lang3-3.4.jar
cp /opt/hadoop/share/hadoop/yarn/timelineservice/lib/commons-lang-2.6.jar /opt/sqoop/lib

# update envvars.sh
cat >> /opt/envvars.sh << EOF
# Sqoop
export SQOOP_HOME=/opt/sqoop
export PATH=\${PATH}:\${SQOOP_HOME}/bin

EOF

cat /opt/envvars.sh

mkdir: cannot create directory '/opt/pkgs': File exists
export JAVA_HOME=/usr/lib/jvm/java-1.8.0-openjdk-amd64
export PDSH_RCMD_TYPE=ssh
export HADOOP_HOME=/opt/hadoop
export HADOOP_VERSION=3.3.6
export HADOOP_COMMON_HOME=${HADOOP_HOME}
export HADOOP_CONF_DIR=${HADOOP_HOME}/etc/hadoop
export HADOOP_HDFS_HOME=${HADOOP_HOME}
export HADOOP_MAPRED_HOME=${HADOOP_HOME}
export HADOOP_YARN_HOME=${HADOOP_HOME}
export PATH=${PATH}:${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin
# Spark
export SPARK_HOME=/opt/spark
export PYSPARK_PYTHON=python3
export PYSPARK_DRIVER_PYTHON=python3
export PYTHONIOENCODING=utf8
export PATH=${PATH}:${SPARK_HOME}/bin

# Hive
export HIVE_HOME=/opt/hive
export PATH=${PATH}:${HIVE_HOME}/bin

# Sqoop
export SQOOP_HOME=/opt/sqoop
export PATH=${PATH}:${SQOOP_HOME}/bin



### Mysql-connector

- https://dev.mysql.com/downloads/connector/j/

In [58]:
%%dockerexec hadoop

# Download package
cd /opt/pkgs
wget -q -c https://downloads.mysql.com/archives/get/p/3/file/mysql-connector-j_8.0.33-1ubuntu20.04_all.deb
    
sudo dpkg -i mysql-connector-j_8.0.33-1ubuntu20.04_all.deb

cp /usr/share/java/mysql-connector-java-8.0.33.jar /opt/sqoop/bin

(Reading database ... 42464 files and directories currently installed.)
Preparing to unpack mysql-connector-j_8.0.33-1ubuntu20.04_all.deb ...
Unpacking mysql-connector-j (8.0.33-1ubuntu20.04) over (8.0.33-1ubuntu20.04) ...
Setting up mysql-connector-j (8.0.33-1ubuntu20.04) ...


## Consultas na base de dados original (airlines)

### Airlines - Consulta 1

Quais vôos são domésticos do Brasil (origem e destino dentro do país)?

In [89]:
%%dockerwrite hadoop /opt/src/spark/airlines_query_1.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col

def main():
    # Inicializar Spark Session
    spark = SparkSession.builder.appName("AirlinesSparkSQL").getOrCreate()

    # consulta SQL
    query = """
        WITH brazil_airports AS (
            SELECT p.airport_id
            FROM air_airports p
            JOIN air_airports_geo g ON g.airport_id = p.airport_id
            WHERE g.country = 'BRAZIL'
        )
        SELECT
            f.flightno AS flight,
            l.airline_name AS airline
        FROM air_flights_schedules f
        JOIN air_airlines l ON f.airline_id = l.airline_id
        WHERE f.from_airport_id IN (SELECT airport_id FROM brazil_airports)
            AND f.to_airport_id IN (SELECT airport_id FROM brazil_airports)
    """

    # Carregar informações de leitura do banco de dados mySQL
    df = spark.read.format('jdbc').option('driver', 'com.mysql.jdbc.Driver').option('url', 'jdbc:mysql://hadoop/airlines').option('user', 'hadoop').option('query', query).load()

    # Exibir resultados
    df.show(30)

    spark.stop()

if __name__ == "__main__":
    main()

Successfully copied 3.07kB to hadoop:/opt/src/spark/airlines_query_1.py


In [90]:
%%dockerexec hadoop

source /opt/envvars.sh

cd /opt/src/spark

spark-submit --master local --jars $SQOOP_HOME/bin/mysql-connector-java-8.0.33.jar airlines_query_1.py # 2> /dev/null
# spark-submit --master yarn --jars $SQOOP_HOME/bin/mysql-connector-java-8.0.33.jar airlines_query_1.py # 2> /dev/null

25/09/08 17:52:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/08 17:52:46 INFO SparkContext: Running Spark version 3.5.6
25/09/08 17:52:46 INFO SparkContext: OS info Linux, 6.12.10-76061203-generic, amd64
25/09/08 17:52:46 INFO SparkContext: Java version 1.8.0_452
25/09/08 17:52:46 INFO ResourceUtils: ==============================================================
25/09/08 17:52:46 INFO ResourceUtils: No custom resources configured for spark.driver.
25/09/08 17:52:46 INFO ResourceUtils: ==============================================================
25/09/08 17:52:46 INFO SparkContext: Submitted application: AirlinesSparkSQL
25/09/08 17:52:46 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: 

### Airlines - Consulta 2

Quais são os 5 aeroportos do mundo que mais recebem vôos às quintas-feiras?

In [138]:
%%dockerwrite hadoop /opt/src/spark/airlines_query_2.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col

def main():
    # Inicializar Spark Session
    spark = SparkSession.builder.appName("AirlinesSparkSQL").getOrCreate()

    # consulta SQL
    query = """
        SELECT
            COUNT(*) AS inbound_flights,
            ap.icao AS icao_code,
            g.city AS city,
            g.country AS country
        FROM air_airports ap
        JOIN air_airports_geo g ON ap.airport_id = g.airport_id
        JOIN air_flights_schedules fs ON fs.to_airport_id = ap.airport_id
        WHERE fs.thursday = 1
        GROUP BY icao_code, city, country
        ORDER BY inbound_flights DESC
        LIMIT 5
    """

    # Carregar informações de leitura do banco de dados mySQL
    df = spark.read.format('jdbc').option('driver', 'com.mysql.jdbc.Driver').option('url', 'jdbc:mysql://hadoop/airlines').option('user', 'hadoop').option('query', query).load()

    # Exibir resultados
    df.show()

    spark.stop()

if __name__ == "__main__":
    main()

Successfully copied 3.07kB to hadoop:/opt/src/spark/airlines_query_2.py


In [139]:
%%dockerexec hadoop

source /opt/envvars.sh

cd /opt/src/spark

spark-submit --master local --jars $SQOOP_HOME/bin/mysql-connector-java-8.0.33.jar airlines_query_2.py # 2> /dev/null
# spark-submit --master yarn --jars $SQOOP_HOME/bin/mysql-connector-java-8.0.33.jar airlines_query_2.py # 2> /dev/null

25/09/08 18:58:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/08 18:58:26 INFO SparkContext: Running Spark version 3.5.6
25/09/08 18:58:26 INFO SparkContext: OS info Linux, 6.12.10-76061203-generic, amd64
25/09/08 18:58:26 INFO SparkContext: Java version 1.8.0_452
25/09/08 18:58:26 INFO ResourceUtils: ==============================================================
25/09/08 18:58:26 INFO ResourceUtils: No custom resources configured for spark.driver.
25/09/08 18:58:26 INFO ResourceUtils: ==============================================================
25/09/08 18:58:26 INFO SparkContext: Submitted application: AirlinesSparkSQL
25/09/08 18:58:26 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: 

### Airlines - Consulta 3

Afim de realizar uma campanha publicitária, uma agência de viagem quer o nome completo e o e-mail dos 20 passageiros que mais gastaram com passagens aéreas.

In [128]:
%%dockerwrite hadoop /opt/src/spark/airlines_query_3.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col

def main():
    # Inicializar Spark Session
    spark = SparkSession.builder.appName("AirlinesSparkSQL").getOrCreate()

    # consulta SQL
    query = """
        SELECT
            pas.passenger_id,
            pas.firstname,
            pas.lastname,
            pd.emailaddress AS email,
            SUM(b.price) AS total_spent
        FROM air_bookings b
        JOIN air_passengers pas ON b.passenger_id = pas.passenger_id
        JOIN air_passengers_details pd ON pd.passenger_id = pas.passenger_id
        GROUP BY pas.passenger_id, pas.firstname, pas.lastname, email
        ORDER BY total_spent DESC
        LIMIT 20
    """

    # Carregar informações de leitura do banco de dados mySQL
    df = spark.read.format('jdbc').option('driver', 'com.mysql.jdbc.Driver').option('url', 'jdbc:mysql://hadoop/airlines').option('user', 'hadoop').option('query', query).load()

    # Exibir resultados
    df.show()

    spark.stop()

if __name__ == "__main__":
    main()

Successfully copied 3.07kB to hadoop:/opt/src/spark/airlines_query_3.py


In [129]:
%%dockerexec hadoop

source /opt/envvars.sh

cd /opt/src/spark

spark-submit --master local --jars $SQOOP_HOME/bin/mysql-connector-java-8.0.33.jar airlines_query_3.py # 2> /dev/null
# spark-submit --master yarn --jars $SQOOP_HOME/bin/mysql-connector-java-8.0.33.jar airlines_query_3.py # 2> /dev/null

25/09/08 18:36:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/08 18:36:26 INFO SparkContext: Running Spark version 3.5.6
25/09/08 18:36:26 INFO SparkContext: OS info Linux, 6.12.10-76061203-generic, amd64
25/09/08 18:36:26 INFO SparkContext: Java version 1.8.0_452
25/09/08 18:36:26 INFO ResourceUtils: ==============================================================
25/09/08 18:36:26 INFO ResourceUtils: No custom resources configured for spark.driver.
25/09/08 18:36:26 INFO ResourceUtils: ==============================================================
25/09/08 18:36:26 INFO SparkContext: Submitted application: AirlinesSparkSQL
25/09/08 18:36:26 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: 

## Setup nova base de dados ()

### Nova base - Consulta 1

### Nova base - Consulta 2

### Nova base - Consulta 3

### Cleanup and stop services

In [26]:
%%dockerexec hadoop

source /opt/envvars.sh

# rm airlines.java

# Stopping mysql
sudo service mysql stop

 * Stopping MySQL database server mysqld
   ...done.
